In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

from src.data.dataset import PreferenceDataset, PreferenceCollator, create_dataloader


MODEL_NAME = "gpt2"
MAX_LENGTH = 512
SAMPLE_SIZE = 20


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Load only 20 examples
if SAMPLE_SIZE is not None:
    raw_dataset = load_dataset(
        "Anthropic/hh-rlhf",
        split=f"train[:{SAMPLE_SIZE}]",
    )
else:
    raw_dataset = load_dataset(
        "Anthropic/hh-rlhf",
        split="train",
    )


dataset = PreferenceDataset(
    data=raw_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# ------------------------------------------------------------
# Take a few individual examples
# ------------------------------------------------------------

examples = [
    dataset[0],
    dataset[1],
    dataset[2],
]


# ------------------------------------------------------------
# Create collator
# ------------------------------------------------------------

collator = PreferenceCollator(
    pad_token_id=tokenizer.pad_token_id
)


# ------------------------------------------------------------
# Create a batch manually
# ------------------------------------------------------------

batch = collator(examples)


# ------------------------------------------------------------
# Inspect batch
# ------------------------------------------------------------

print("\nChosen input IDs:")
print(batch["chosen_input_ids"].shape)

print("\nChosen attention mask:")
print(batch["chosen_attention_mask"].shape)

print("\nRejected input IDs:")
print(batch["rejected_input_ids"].shape)

print("\nRejected attention mask:")
print(batch["rejected_attention_mask"].shape)


# ------------------------------------------------------------
# Inspect actual attention masks
# ------------------------------------------------------------

print("\nChosen attention mask:")
print(batch["chosen_attention_mask"])

print("\nRejected attention mask:")
print(batch["rejected_attention_mask"])


Chosen input IDs:
torch.Size([3, 202])

Chosen attention mask:
torch.Size([3, 202])

Rejected input IDs:
torch.Size([3, 196])

Rejected attention mask:
torch.Size([3, 196])

Chosen attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 

In [2]:
dataloader = create_dataloader(
    dataset=dataset,
    tokenizer=tokenizer,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

In [3]:
batch = next(iter(dataloader))

In [4]:
print("\nBatch shapes:")

print(
    "Chosen input IDs:",
    batch["chosen_input_ids"].shape,
)

print(
    "Chosen attention mask:",
    batch["chosen_attention_mask"].shape,
)

print(
    "Rejected input IDs:",
    batch["rejected_input_ids"].shape,
)

print(
    "Rejected attention mask:",
    batch["rejected_attention_mask"].shape,
)


Batch shapes:
Chosen input IDs: torch.Size([4, 202])
Chosen attention mask: torch.Size([4, 202])
Rejected input IDs: torch.Size([4, 196])
Rejected attention mask: torch.Size([4, 196])


In [5]:
print("\nChosen sequence lengths from attention mask:")

print(
    batch["chosen_attention_mask"].sum(dim=1)
)

print("\nRejected sequence lengths from attention mask:")

print(
    batch["rejected_attention_mask"].sum(dim=1)
)


Chosen sequence lengths from attention mask:
tensor([202, 107,  53, 101])

Rejected sequence lengths from attention mask:
tensor([196, 117, 181, 106])
